In [1]:
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

warnings.filterwarnings("ignore")


# ============================================================
# 1. PARAMÈTRES
# ============================================================

INPUT_FILE = "dataset_finale_long.csv"
OUTPUT_FILE = "benchmark_arima_prophet_2024.csv"

COL_ZONE = "Zone géographique"
COL_YEAR = "Annee"
COL_POP = "Population"

TRAIN_START = 2004
TRAIN_END = 2023
TEST_YEAR = 2024

EPSILON = 1e-9


# ============================================================
# 2. CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df.columns = df.columns.astype(str).str.strip()

# Gestion automatique si la colonne s'appelle "Année"
if "Année" in df.columns and "Annee" not in df.columns:
    df = df.rename(columns={"Année": "Annee"})

required_cols = [COL_ZONE, COL_YEAR, COL_POP]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Colonnes manquantes dans le dataset : {missing_cols}")

df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce").astype("Int64")
df[COL_POP] = pd.to_numeric(df[COL_POP], errors="coerce").astype("float64")

df = df.dropna(subset=[COL_ZONE, COL_YEAR, COL_POP]).copy()
df[COL_YEAR] = df[COL_YEAR].astype(int)

# Garder l'ordre original des zones comme dans le fichier source
df["ordre_original"] = pd.factorize(df[COL_ZONE])[0]

df = df.sort_values([COL_ZONE, COL_YEAR]).reset_index(drop=True)


# ============================================================
# 3. FONCTIONS DE MÉTRIQUES
# ============================================================

def compute_metrics(y_true, y_pred):
    """
    Calcule MAE, RMSE et MAPE pour une prédiction unique.
    """
    error = y_true - y_pred

    mae = abs(error)
    rmse = np.sqrt(error ** 2)

    if abs(y_true) < EPSILON:
        mape = np.nan
    else:
        mape = abs(error / y_true) * 100

    return mae, rmse, mape


# ============================================================
# 4. MODÈLE ARIMA
# ============================================================

def forecast_arima(train_series):
    """
    Teste plusieurs configurations ARIMA simples.
    Garde le modèle avec le meilleur AIC.
    Retourne uniquement la prédiction 2024.
    """

    candidate_orders = [
        (0, 1, 0),
        (1, 1, 0),
        (0, 1, 1),
        (1, 1, 1),
        (2, 1, 0),
        (0, 1, 2),
        (2, 1, 1),
        (1, 1, 2)
    ]

    best_aic = np.inf
    best_model = None

    for order in candidate_orders:
        try:
            model = ARIMA(
                train_series,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = model.fit()

            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_model = fitted_model

        except Exception:
            continue

    if best_model is None:
        return np.nan

    forecast = best_model.forecast(steps=1)

    return float(forecast.iloc[0])


# ============================================================
# 5. MODÈLE PROPHET
# ============================================================

def forecast_prophet(train_df):
    """
    Entraîne Prophet sur 2004-2023 et prédit 2024.
    Données annuelles : pas de saisonnalité.
    """

    prophet_df = train_df[[COL_YEAR, COL_POP]].copy()

    prophet_df = prophet_df.rename(columns={
        COL_YEAR: "ds",
        COL_POP: "y"
    })

    prophet_df["ds"] = pd.to_datetime(
        prophet_df["ds"].astype(str) + "-01-01"
    )

    try:
        model = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode="additive"
        )

        model.fit(prophet_df)

        future = pd.DataFrame({
            "ds": [pd.to_datetime(f"{TEST_YEAR}-01-01")]
        })

        forecast = model.predict(future)

        pred_2024 = float(forecast["yhat"].iloc[0])

        return pred_2024

    except Exception:
        return np.nan


# ============================================================
# 6. BOUCLE PRINCIPALE PAR ZONE
# ============================================================

results = []

zones_order = (
    df[[COL_ZONE, "ordre_original"]]
    .drop_duplicates()
    .sort_values("ordre_original")[COL_ZONE]
    .tolist()
)

for zone in zones_order:
    df_zone = df[df[COL_ZONE] == zone].copy()
    df_zone = df_zone.sort_values(COL_YEAR)

    ordre_zone = int(df_zone["ordre_original"].iloc[0])

    train_df = df_zone[
        (df_zone[COL_YEAR] >= TRAIN_START) &
        (df_zone[COL_YEAR] <= TRAIN_END)
    ].copy()

    test_df = df_zone[df_zone[COL_YEAR] == TEST_YEAR].copy()

    if train_df.empty or test_df.empty:
        continue

    if len(train_df) < 5:
        continue

    y_true_2024 = float(test_df[COL_POP].iloc[0])

    train_series = train_df.set_index(COL_YEAR)[COL_POP].astype(float)

    # ---------------------------
    # ARIMA
    # ---------------------------

    pred_arima = forecast_arima(train_series)

    if np.isfinite(pred_arima):
        pred_arima = round(pred_arima)
        mae_arima, rmse_arima, mape_arima = compute_metrics(
            y_true_2024,
            pred_arima
        )
    else:
        mae_arima, rmse_arima, mape_arima = np.nan, np.nan, np.nan

    # ---------------------------
    # Prophet
    # ---------------------------

    pred_prophet = forecast_prophet(train_df)

    if np.isfinite(pred_prophet):
        pred_prophet = round(pred_prophet)
        mae_prophet, rmse_prophet, mape_prophet = compute_metrics(
            y_true_2024,
            pred_prophet
        )
    else:
        mae_prophet, rmse_prophet, mape_prophet = np.nan, np.nan, np.nan

    # ---------------------------
    # Sélection automatique
    # ---------------------------

    if np.isnan(mape_arima) and np.isnan(mape_prophet):
        best_model = "Aucun modèle valide"

    elif np.isnan(mape_arima):
        best_model = "Prophet"

    elif np.isnan(mape_prophet):
        best_model = "ARIMA"

    elif mape_arima <= mape_prophet:
        best_model = "ARIMA"

    else:
        best_model = "Prophet"

    results.append({
        "ordre_original": ordre_zone,
        "Zone géographique": zone,
        "Population réelle 2024": round(y_true_2024),

        "Prediction_ARIMA_2024": pred_arima,
        "MAE_ARIMA": mae_arima,
        "RMSE_ARIMA": rmse_arima,
        "MAPE_ARIMA_%": mape_arima,

        "Prediction_Prophet_2024": pred_prophet,
        "MAE_Prophet": mae_prophet,
        "RMSE_Prophet": rmse_prophet,
        "MAPE_Prophet_%": mape_prophet,

        "Modèle retenu": best_model
    })


# ============================================================
# 7. TABLEAU FINAL DE SYNTHÈSE
# ============================================================

benchmark_df = pd.DataFrame(results)

benchmark_df = benchmark_df.sort_values(
    by="ordre_original"
).reset_index(drop=True)

benchmark_df = benchmark_df.drop(columns=["ordre_original"])


# ============================================================
# 8. ARRONDIS DES VALEURS
# ============================================================

cols_entieres = [
    "Population réelle 2024",
    "Prediction_ARIMA_2024",
    "Prediction_Prophet_2024",
    "MAE_ARIMA",
    "RMSE_ARIMA",
    "MAE_Prophet",
    "RMSE_Prophet"
]

for col in cols_entieres:
    benchmark_df[col] = benchmark_df[col].round(0).astype("Int64")

cols_mape = [
    "MAPE_ARIMA_%",
    "MAPE_Prophet_%"
]

for col in cols_mape:
    benchmark_df[col] = benchmark_df[col].round(3)


# ============================================================
# 9. EXPORT
# ============================================================

benchmark_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)


# ============================================================
# 10. RÉSUMÉ
# ============================================================

print("Benchmark terminé avec succès.")
print(f"Nombre de zones évaluées : {len(benchmark_df)}")
print(f"Fichier généré : {OUTPUT_FILE}")

print("\nRépartition des modèles retenus :")
print(benchmark_df["Modèle retenu"].value_counts())

print("\nAperçu du tableau final :")
print(benchmark_df.head())

c:\Users\LENOVO\.conda\envs\pfe_hcp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
16:50:52 - cmdstanpy - INFO - Chain [1] start processing
16:50:52 - cmdstanpy - INFO - Chain [1] done processing
16:50:52 - cmdstanpy - INFO - Chain [1] start processing
16:50:53 - cmdstanpy - INFO - Chain [1] done processing
16:50:53 - cmdstanpy - INFO - Chain [1] start processing
16:50:53 - cmdstanpy - INFO - Chain [1] done processing
16:50:54 - cmdstanpy - INFO - Chain [1] start processing
16:50:54 - cmdstanpy - INFO - Chain [1] done processing
16:50:55 - cmdstanpy - INFO - Chain [1] start processing
16:50:55 - cmdstanpy - INFO - Chain [1] done processing
16:50:55 - cmdstanpy - INFO - Chain [1] start processing
16:50:56 - cmdstanpy - INFO - Chain [1] done processing
16:50

Benchmark terminé avec succès.
Nombre de zones évaluées : 177
Fichier généré : benchmark_arima_prophet_2024.csv

Répartition des modèles retenus :
Modèle retenu
ARIMA      124
Prophet     53
Name: count, dtype: int64

Aperçu du tableau final :
           Zone géographique  Population réelle 2024  Prediction_ARIMA_2024  \
0                   National                36828330               36826091   
1  Tanger-Tétouan-Al Hoceima                 4030222                4031066   
2          Al Hoceima (Prov)                  371527                 371609   
3                 Al Hoceima                   50225                  50251   
4               Bni Bouayach                   20013                  20032   

   MAE_ARIMA  RMSE_ARIMA  MAPE_ARIMA_%  Prediction_Prophet_2024  MAE_Prophet  \
0       2239        2239         0.006                 36844139        15809   
1        844         844         0.021                  4032535         2313   
2         82          82         0.022   

In [2]:
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

warnings.filterwarnings("ignore")


# ============================================================
# 1. PARAMÈTRES
# ============================================================

WIDE_FILE = "dataset_finale.csv"
DATA_FILE = "dataset_finale_long.csv"
BENCHMARK_FILE = "benchmark_arima_prophet_2024.csv"

OUTPUT_WIDE_FILE = "projections_2025_2040_wide.csv"
OUTPUT_LONG_FILE = "projections_2025_2040_long.csv"

COL_ZONE = "Zone géographique"
COL_YEAR = "Annee"
COL_POP = "Population"
COL_MODEL = "Modèle retenu"

HIST_START = 2004
HIST_END = 2024

FORECAST_START = 2025
FORECAST_END = 2040
FORECAST_YEARS = list(range(FORECAST_START, FORECAST_END + 1))


# ============================================================
# 2. CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv(
    DATA_FILE,
    sep=";",
    encoding="utf-8-sig"
)

benchmark = pd.read_csv(
    BENCHMARK_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df_wide_order = pd.read_csv(
    WIDE_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df.columns = df.columns.astype(str).str.strip()
benchmark.columns = benchmark.columns.astype(str).str.strip()
df_wide_order.columns = df_wide_order.columns.astype(str).str.strip()

# Gestion automatique si la colonne s'appelle "Année"
if "Année" in df.columns and "Annee" not in df.columns:
    df = df.rename(columns={"Année": "Annee"})

required_data_cols = [COL_ZONE, COL_YEAR, COL_POP]
required_benchmark_cols = [COL_ZONE, COL_MODEL]
required_wide_cols = [COL_ZONE]

missing_data = [c for c in required_data_cols if c not in df.columns]
missing_benchmark = [c for c in required_benchmark_cols if c not in benchmark.columns]
missing_wide = [c for c in required_wide_cols if c not in df_wide_order.columns]

if missing_data:
    raise ValueError(f"Colonnes manquantes dans {DATA_FILE} : {missing_data}")

if missing_benchmark:
    raise ValueError(f"Colonnes manquantes dans {BENCHMARK_FILE} : {missing_benchmark}")

if missing_wide:
    raise ValueError(f"Colonnes manquantes dans {WIDE_FILE} : {missing_wide}")


# ============================================================
# 3. NETTOYAGE DES DONNÉES
# ============================================================

df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce").astype("Int64")
df[COL_POP] = pd.to_numeric(df[COL_POP], errors="coerce").astype("float64")

df = df.dropna(subset=[COL_ZONE, COL_YEAR, COL_POP]).copy()
df[COL_YEAR] = df[COL_YEAR].astype(int)

df[COL_ZONE] = df[COL_ZONE].astype(str).str.strip()
benchmark[COL_ZONE] = benchmark[COL_ZONE].astype(str).str.strip()
benchmark[COL_MODEL] = benchmark[COL_MODEL].astype(str).str.strip()
df_wide_order[COL_ZONE] = df_wide_order[COL_ZONE].astype(str).str.strip()

df = df.sort_values([COL_ZONE, COL_YEAR]).reset_index(drop=True)


# ============================================================
# 4. ORDRE ORIGINAL DES ZONES DEPUIS dataset_finale.csv
# ============================================================

ordre_zones = df_wide_order[[COL_ZONE]].copy()
ordre_zones = ordre_zones.drop_duplicates().reset_index(drop=True)
ordre_zones["ordre_original"] = range(len(ordre_zones))


# ============================================================
# 5. FONCTION ARIMA
# ============================================================

def forecast_arima_full(train_series, steps):
    """
    Entraîne ARIMA sur toute la série 2004-2024.
    Sélection du meilleur ordre par AIC parmi plusieurs modèles simples.
    """

    candidate_orders = [
        (0, 1, 0),
        (1, 1, 0),
        (0, 1, 1),
        (1, 1, 1),
        (2, 1, 0),
        (0, 1, 2),
        (2, 1, 1),
        (1, 1, 2)
    ]

    best_aic = np.inf
    best_model = None

    for order in candidate_orders:
        try:
            model = ARIMA(
                train_series,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted = model.fit()

            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_model = fitted

        except Exception:
            continue

    if best_model is None:
        return [np.nan] * steps

    forecast = best_model.forecast(steps=steps)

    return forecast.values.tolist()


# ============================================================
# 6. FONCTION PROPHET
# ============================================================

def forecast_prophet_full(train_df, future_years):
    """
    Entraîne Prophet sur toute la série 2004-2024.
    Prévoit de 2025 jusqu'à 2040.
    Données annuelles : pas de saisonnalité.
    """

    prophet_df = train_df[[COL_YEAR, COL_POP]].copy()

    prophet_df = prophet_df.rename(columns={
        COL_YEAR: "ds",
        COL_POP: "y"
    })

    prophet_df["ds"] = pd.to_datetime(
        prophet_df["ds"].astype(str) + "-01-01"
    )

    try:
        model = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode="additive"
        )

        model.fit(prophet_df)

        future = pd.DataFrame({
            "ds": [pd.to_datetime(f"{year}-01-01") for year in future_years]
        })

        forecast = model.predict(future)

        return forecast["yhat"].values.tolist()

    except Exception:
        return [np.nan] * len(future_years)


# ============================================================
# 7. PRÉPARATION DU TABLEAU DES MODÈLES RETENUS
# ============================================================

benchmark_models = benchmark[[COL_ZONE, COL_MODEL]].copy()

df_model = ordre_zones.merge(
    benchmark_models,
    on=COL_ZONE,
    how="left"
)

missing_model = df_model[df_model[COL_MODEL].isna()].copy()

if not missing_model.empty:
    print("Attention : certaines zones n'ont pas de modèle retenu dans le benchmark.")
    print("Elles seront ignorées.")
    print(missing_model[[COL_ZONE]].head(20))


# ============================================================
# 8. BOUCLE DE PROJECTION 2025-2040
# ============================================================

projection_rows = []

for _, row in df_model.iterrows():
    zone = row[COL_ZONE]
    modele_retenu = row[COL_MODEL]

    if pd.isna(modele_retenu):
        continue

    df_zone = df[
        (df[COL_ZONE] == zone) &
        (df[COL_YEAR] >= HIST_START) &
        (df[COL_YEAR] <= HIST_END)
    ].copy()

    df_zone = df_zone.sort_values(COL_YEAR)

    if len(df_zone) < 5:
        continue

    train_series = df_zone.set_index(COL_YEAR)[COL_POP].astype(float)

    modele_retenu_clean = str(modele_retenu).strip().lower()

    if modele_retenu_clean == "arima":
        preds = forecast_arima_full(
            train_series=train_series,
            steps=len(FORECAST_YEARS)
        )

    elif modele_retenu_clean == "prophet":
        preds = forecast_prophet_full(
            train_df=df_zone,
            future_years=FORECAST_YEARS
        )

    else:
        preds = [np.nan] * len(FORECAST_YEARS)

    # Sécurité : population non négative
    preds = [
        max(0, pred) if np.isfinite(pred) else np.nan
        for pred in preds
    ]

    # Arrondi final : population = individus
    preds = [
        round(pred) if np.isfinite(pred) else np.nan
        for pred in preds
    ]

    result = {
        "ordre_original": row["ordre_original"],
        COL_ZONE: zone,
        "Modèle utilisé": modele_retenu
    }

    for year, pred in zip(FORECAST_YEARS, preds):
        result[str(year)] = pred

    projection_rows.append(result)


# ============================================================
# 9. CRÉATION FORMAT WIDE
# ============================================================

projections_wide = pd.DataFrame(projection_rows)

projections_wide = projections_wide.sort_values(
    by="ordre_original"
).reset_index(drop=True)

projections_wide = projections_wide.drop(columns=["ordre_original"])

year_cols = [str(y) for y in FORECAST_YEARS]

for col in year_cols:
    projections_wide[col] = projections_wide[col].round(0).astype("Int64")


# ============================================================
# 10. CRÉATION FORMAT LONG
# ============================================================

projections_long = projections_wide.melt(
    id_vars=[COL_ZONE, "Modèle utilisé"],
    value_vars=year_cols,
    var_name="Annee",
    value_name="Population projetée"
)

projections_long["Annee"] = projections_long["Annee"].astype(int)
projections_long["Population projetée"] = (
    projections_long["Population projetée"]
    .round(0)
    .astype("Int64")
)

# Garder l'ordre original : zone puis année
projections_long = projections_long.sort_values(
    by=[COL_ZONE, "Annee"]
).reset_index(drop=True)

# Si tu veux garder l'ordre original exact du fichier source dans le long aussi :
ordre_map = {
    zone: i
    for i, zone in enumerate(projections_wide[COL_ZONE].tolist())
}

projections_long["ordre_original"] = projections_long[COL_ZONE].map(ordre_map)

projections_long = projections_long.sort_values(
    by=["ordre_original", "Annee"]
).reset_index(drop=True)

projections_long = projections_long.drop(columns=["ordre_original"])


# ============================================================
# 11. EXPORT DES DEUX FORMATS
# ============================================================

projections_wide.to_csv(
    OUTPUT_WIDE_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

projections_long.to_csv(
    OUTPUT_LONG_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)


# ============================================================
# 12. RÉSUMÉ
# ============================================================

print("Projection terminée avec succès.")
print(f"Horizon de projection : {FORECAST_START}-{FORECAST_END}")

print(f"\nNombre de zones projetées : {len(projections_wide)}")

print("\nFichiers générés :")
print(f"- {OUTPUT_WIDE_FILE}")
print(f"- {OUTPUT_LONG_FILE}")

print("\nRépartition des modèles utilisés :")
print(projections_wide["Modèle utilisé"].value_counts())

print("\nAperçu format wide :")
print(projections_wide.head())

print("\nAperçu format long :")
print(projections_long.head(20))

14:48:04 - cmdstanpy - INFO - Chain [1] start processing
14:48:06 - cmdstanpy - INFO - Chain [1] done processing
14:48:10 - cmdstanpy - INFO - Chain [1] start processing
14:48:10 - cmdstanpy - INFO - Chain [1] done processing
14:48:11 - cmdstanpy - INFO - Chain [1] start processing
14:48:11 - cmdstanpy - INFO - Chain [1] done processing
14:48:14 - cmdstanpy - INFO - Chain [1] start processing
14:48:14 - cmdstanpy - INFO - Chain [1] done processing
14:48:14 - cmdstanpy - INFO - Chain [1] start processing
14:48:15 - cmdstanpy - INFO - Chain [1] done processing
14:48:19 - cmdstanpy - INFO - Chain [1] start processing
14:48:20 - cmdstanpy - INFO - Chain [1] done processing
14:48:22 - cmdstanpy - INFO - Chain [1] start processing
14:48:22 - cmdstanpy - INFO - Chain [1] done processing
14:48:24 - cmdstanpy - INFO - Chain [1] start processing
14:48:25 - cmdstanpy - INFO - Chain [1] done processing
14:48:25 - cmdstanpy - INFO - Chain [1] start processing
14:48:26 - cmdstanpy - INFO - Chain [1]

Projection terminée avec succès.
Horizon de projection : 2025-2040

Nombre de zones projetées : 177

Fichiers générés :
- projections_2025_2040_wide.csv
- projections_2025_2040_long.csv

Répartition des modèles utilisés :
Modèle utilisé
ARIMA      124
Prophet     53
Name: count, dtype: int64

Aperçu format wide :
           Zone géographique Modèle utilisé      2025      2026      2027  \
0                   National          ARIMA  37114597  37394683  37668659   
1  Tanger-Tétouan-Al Hoceima          ARIMA   4076436   4122518   4168497   
2          Al Hoceima (Prov)          ARIMA    368240    364908    361532   
3                 Al Hoceima          ARIMA     49362     48472     47554   
4               Bni Bouayach          ARIMA     20129     20241     20349   

       2028      2029      2030      2031      2032      2033      2034  \
0  37936658  38198813  38455248  38706091  38951461  39191480  39426263   
1   4214371   4260143   4305812   4351377   4396840   4442201   4487460 